In [4]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

np.random.seed(55)
n = 500
kategori_list = [
    "Elektronik",
    "Fashion",
    "Makanan & Minuman",
    "Kesehatan & Kecantikan",
    "Rumah Tangga",
]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice(
        [25000, 50000, 75000, 100000, 150000], size=n
    ),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

spark = SparkSession.builder.appName("Tugas5_Praktikum").getOrCreate()

df_transaksi = spark.read.csv(
    "transaksi_tugas5.csv", header=True, inferSchema=True
)

df_transaksi = df_transaksi.withColumn(
    "pendapatan", col("unit_terjual") * col("harga_satuan")
)
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("Inisialisasi berhasil!")

Inisialisasi berhasil!


In [5]:
df_total_pendapatan = df_transaksi.groupBy("kota").agg(_sum("pendapatan").alias("total_pendapatan"))

df_bagian_a = df_total_pendapatan.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(col("pencapaian_persen").desc())

df_bagian_a.show()

[Stage 5:>                                                        (0 + 18) / 18]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [6]:
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(_sum("pendapatan").alias("total_pendapatan"))

windowSpec = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

df_bagian_b = df_kategori_kota.withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") == 1) \
    .drop("rank") \
    .orderBy("kota")

df_bagian_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [7]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

df_bagian_c = spark.sql("""
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

df_bagian_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



In [ ]:
Berdasarkan hasil analisis data performa cabang platform e-commerce:

1. **Cabang Berkinerja Paling Baik:*
Cabang **Purworejo** (PIC: Fitri) menunjukkan kinerja terbaik karena menjadi
satu-satunya cabang yang berhasil melampaui target bulanan dengan pencapaian 
hingga **152,17%** (total pendapatan Rp45.650.000 dari target Rp30.000.000). 
Tingginya performa ini didorong oleh volume transaksi terbanyak dibanding kota 
lain, yaitu sebanyak **116 transaksi**, dengan kontribusi pendapatan terbesar 
berasal dari kategori **Kesehatan & Kecantikan** sebesar Rp10.075.000.

2. **Cabang yang Paling Perlu Perhatian Manajemen:**
   Cabang **Semarang** (PIC: Sari) paling membutuhkan perhatian khusus dari 
manajemen karena mencatatkan tingkat pencapaian target paling rendah, yaitu 
hanya **69,41%** (total pendapatan Rp38.175.000 dari target Rp55.000.000). 
Selain itu, cabang **Magelang** (PIC: Rani) juga memerlukan evaluasi karena 
memiliki jumlah transaksi paling sedikit (**86 transaksi**) dengan pencapaian 
target yang rendah sebesar **70,33%** (Rp31.650.000 dari target Rp45.000.000).
Evaluasi strategi pemasaran dan penyesuaian target perlu difokuskan pada kedua cabang ini.